# Görev — Katman-başı Linearize Maliyeti Haritası (yoğunluk kararları için)

**Soru.** Hangi attention katmanları O(1) belleğe *ucuza* çevrilebiliyor, hangileri
*pahalı*? Bu harita, graft yoğunluğunu (g) körlemesine "tek indeksliler" yerine
**ilkeli** seçmeyi sağlar → §22a uçurumunun altında kalarak daha çok katman
linearize etme umudu (RESULTS §23 Pareto'sunu ileri itmek).

## Yöntemin çekirdek içgörüsü (neden ucuz ve temiz)
`teacher_forcing` modunda kod ileriye **öğretmen çıktısını** geçirir; yani her
graft'lı katman, *doğru (öğretmen) gizli-durum* girdisi üzerinde **bağımsız**
eğitilir — yukarı akıştaki diğer graft'lı katmanlardan etkilenmez (grafting.py
docstring §4). Sonuç: **28 katmanın hepsini tek koşuda** graft'layıp her birinin
yakınsamış rekonstrüksiyon hatasını **paralel** okuyabiliriz. Tek Stage-1 koşusu
= tam harita.

## Ölçüt: Normalize MSE (NMSE), ham MSE DEĞİL
Ham MSE katmanlar arası kıyaslanamaz (derin katmanların aktivasyon normu büyük →
MSE büyük görünür, adil değil). Bu yüzden katman i için:
  **NMSE_i = ||öğrenci − öğretmen||² / ||öğretmen||²**
(1.0 = model çıktının tümünü ıskalıyor; 0 = mükemmel taklit.) Öğretmen enerjisi,
teacher_forcing'de modül çıktısı = öğretmen çıktısı olduğundan aynı ileri-geçişte
forward-hook ile ölçülür.

## ÖN-KAYITLI KRİTERLER (koşudan ÖNCE yazıldı — AGENTS.md kural #1)
1. **Birincil çıktı:** 28 katmanın NMSE haritası (bar grafik + JSON).
2. **H1 — seçim manevra alanı:** En ucuz 6 katmanın ortalama NMSE'si, mevcut
   referans set [3,7,11,15,19,23]'ün ortalamasından **belirgin** (≥%20) düşükse,
   ilkeli seçimin manevra alanı VAR → daha yüksek yoğunluk için umut. Değilse,
   "tek indeksliler zaten iyi bir seçim" sonucu (yine bilgi).
3. **H2 — bilinen ön-sav (metrik geçerlilik kontrolü):** İlk (0) ve son (27)
   katman en pahalılar arasında ÇIKMALI (Mamba-in-Llama full bırakıyor). Çıkarsa
   metrik anlamlı; çıkmazsa metriği sorgula.
4. **DÜRÜST KISIT (kritik):** NMSE bir **PROXY**'dir — tek-katman rekonstrüksiyonu,
   temiz öğretmen girdisinde. Şunları YAKALAMAZ: (a) çok katman birlikte graft
   edilince biriken hata, (b) aşağı-akış PPL duyarlılığı (düşük-NMSE bir katman
   PPL-kritik olabilir, ya da tersi). Bu yüzden harita adayları **sıralar**;
   gerçek kazanç ancak **Faz-B** ile doğrulanır: en-ucuz-K'yı graft'la → tam
   Stage-1+2 → PPL'i §22a odd-index PPL'i ile kıyasla. Bu notebook Faz-A'dır.

**Ortam.** Colab Pro T4. Eğitim ~40-50 dk (28 katman, Stage-1). Checkpoint yok
(tanı koşusu); yalnız NMSE haritası JSON+PNG kaydedilir.


In [ ]:
# --- 1. KURULUM ---
import os, subprocess, sys, glob, time, json, math, urllib.request, zipfile
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
import torch
assert torch.cuda.is_available(), 'GPU YOK! Runtime > T4 GPU.'
_cap=torch.cuda.get_device_capability(0); _arch=f'sm_{_cap[0]}{_cap[1]}'
assert _arch in torch.cuda.get_arch_list(), f'{_arch} desteklenmiyor -> T4 sec (P100 degil).'
DEV='cuda'
BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else ('/content' if os.path.exists('/content') else '.')
REPO = os.path.join(BASE, 'HFP')
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/kayra-hn/HFP.git',REPO], check=True)
else:
    subprocess.run(['git','-C',REPO,'pull'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-U','transformers>=4.46','accelerate'], check=True)
os.chdir(REPO); sys.path.insert(0, REPO)
OUT_DIR = os.path.join(REPO,'docs','assets')
try:
    from google.colab import drive; drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/hfp_bench'
except Exception as e:
    print('Drive yok ->', type(e).__name__)
os.makedirs(OUT_DIR, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0), '| cikti:', OUT_DIR)

In [ ]:
# --- 2. MODEL (fp32) + WT-2 PPL ref + WT-103 egitim korpusu ---
from transformers import AutoModelForCausalLM, AutoTokenizer
MODEL_ID='Qwen/Qwen2.5-1.5B'
def _find_local_model():
    for root in ['/kaggle/input', OUT_DIR, BASE, '/content/drive/MyDrive', '/content']:
        if not root or not os.path.isdir(root): continue
        for cfg in glob.glob(f'{root}/**/config.json', recursive=True):
            d=os.path.dirname(cfg)
            if glob.glob(f'{d}/*.safetensors'): return d
    return None
LOCAL=_find_local_model()
if not LOCAL:
    _tk=os.environ.get('HF_TOKEN')
    try:
        from google.colab import userdata; _tk=_tk or userdata.get('HF_TOKEN')
    except Exception: pass
    assert _tk, 'Model yok + HF_TOKEN yok. Colab: Secrets>HF_TOKEN; Kaggle: Add Input Qwen2.5-1.5B.'
    from huggingface_hub import snapshot_download
    LOCAL=f'{BASE}/qwen_model'
    snapshot_download(repo_id=MODEL_ID, local_dir=LOCAL, token=_tk,
        allow_patterns=['*.json','*.txt','tokenizer.model','*.safetensors','*.safetensors.index.json'], max_workers=2)
print('Model:', LOCAL)
tok=AutoTokenizer.from_pretrained(LOCAL)
model=AutoModelForCausalLM.from_pretrained(LOCAL, torch_dtype=torch.float32).to(DEV).eval()
N_LAYERS=model.config.num_hidden_layers
print(f'{MODEL_ID}: {sum(p.numel() for p in model.parameters())/1e9:.2f}B, {N_LAYERS} katman')

# WT-2 valid (NMSE eval girdisi + PPL ref)
WT2='https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/'
for fn in ('valid.txt',):
    dst=f'{BASE}/wt2_{fn}'
    if not os.path.exists(dst): urllib.request.urlretrieve(WT2+fn, dst)
val_ids=tok(open(f'{BASE}/wt2_valid.txt',encoding='utf-8').read(), return_tensors='pt').input_ids[0]
print('WT-2 valid:', f'{len(val_ids):,} token')

# WT-103 egitim korpusu (S3 raw; xet YOK) — ana notebook ile ayni saglam yol
def _find_wt103():
    for pat in ('wiki.train.raw','wiki.train.tokens','*train*.raw','*train*.tokens'):
        h=glob.glob(f'/kaggle/input/**/{pat}', recursive=True)
        if h: return h[0]
    return None
TRAIN_RAW=_find_wt103(); CORPUS='WT-103 (Kaggle Input)'
if TRAIN_RAW is None:
    _wt=f'{BASE}/wikitext-103-raw/wiki.train.raw'
    if not os.path.exists(_wt):
        zp=f'{BASE}/wt103.zip'
        for u in ['https://wikitext.smerity.com/wikitext-103-raw-v1.zip',
                  'https://s3.amazonaws.com/research.metamind.io/wikitext/wikitext-103-raw-v1.zip']:
            try:
                print('WT-103 iniyor:', u, flush=True)
                req=urllib.request.Request(u, headers={'User-Agent':'Mozilla/5.0','Accept':'*/*'})
                with urllib.request.urlopen(req, timeout=120) as r, open(zp,'wb') as f:
                    while True:
                        b=r.read(1<<20)
                        if not b: break
                        f.write(b)
                if os.path.getsize(zp)>1_000_000: break
            except Exception as e: print(' basarisiz:', type(e).__name__)
        with zipfile.ZipFile(zp) as z: z.extractall(BASE)
    TRAIN_RAW=_wt; CORPUS='WT-103 (S3 raw)'
assert os.path.exists(TRAIN_RAW), 'WT-103 yok -> Kaggle Add Input ile wikitext-103 ekle.'
print('Egitim korpusu:', CORPUS, f'{os.path.getsize(TRAIN_RAW)/1e6:.0f} MB')

In [ ]:
# --- 3. TUM 28 KATMANI GRAFT + PARAMETRELER ---
from hfp.models.grafting import (graft_llama, GraftConfig, set_graft_mode, set_distill_backward,
                                  distill_loss, trainable_parameters, HFPGraftAttention)
# Referans receteyle AYNI primitif (adil): exp + additive + dpfp. Tek fark: TUM katmanlar.
GCFG = GraftConfig(decay_mode='exp', write_rule='additive', key_feature_map='dpfp')
ALL_LAYERS = list(range(N_LAYERS))
REF_LAYERS = [3,7,11,15,19,23]     # mevcut 6-katman referans (RESULTS §22)
SEQ, BATCH, ACCUM, STEPS1, LR1 = 128, 1, 4, 700, 1e-3
EVAL_BATCHES = 24                  # NMSE'yi kac held-out chunk'ta ortala

graft_llama(model, GCFG, layers=ALL_LAYERS)
_grafted=[m for m in model.modules() if isinstance(m, HFPGraftAttention)]
assert len(_grafted)==N_LAYERS, f'{len(_grafted)} != {N_LAYERS}'
print(f'TUM {N_LAYERS} katman graft edildi. Egitilebilir param: '
      f'{sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

def batch_iter(path, seq, batch):
    buf=[]
    while True:
        with open(path, encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                buf.extend(tok(line).input_ids)
                while len(buf) >= batch*(seq+1):
                    ch=buf[:batch*(seq+1)]; buf=buf[batch*(seq+1):]
                    t=torch.tensor(ch).view(batch, seq+1)
                    yield t[:, :-1], t[:, 1:]
it = batch_iter(TRAIN_RAW, SEQ, BATCH)
print('parametreler hazir')

In [ ]:
# --- 4. STAGE-1 TEACHER-FORCING (tum katmanlar bagimsiz; per-katman MSE izlenir) ---
from transformers import get_cosine_schedule_with_warmup
model.config.use_cache=False
opt=torch.optim.AdamW(trainable_parameters(model), lr=LR1, weight_decay=0.01)
sch=get_cosine_schedule_with_warmup(opt, 100, STEPS1)
set_graft_mode(model,'teacher_forcing'); set_distill_backward(model, 1.0/ACCUM)  # per-katman ani backward (VRAM)
model.train()
idx_of={id(m): m.layer_idx for m in _grafted}
traj={}   # step -> {layer: mse}
t0=time.time()
for step in range(1, STEPS1+1):
    for _ in range(ACCUM):
        x,_=next(it); model(x.to(DEV)); _=distill_loss(model)
    torch.nn.utils.clip_grad_norm_(trainable_parameters(model), 1.0)
    opt.step(); sch.step(); opt.zero_grad(set_to_none=True)
    if step%100==0 or step<=2:
        snap={m.layer_idx: float(m.last_distill_loss) for m in _grafted}
        traj[step]=snap
        mn=min(snap.values()); mx=max(snap.values())
        print(f'S1 {step}/{STEPS1}  mse[min {mn:.4f} / max {mx:.4f}]  {(time.time()-t0)/60:.0f}dk', flush=True)
print('Stage-1 tamam.')

In [ ]:
# --- 5. NMSE OLCUMU (held-out; MSE + ogretmen enerjisi ayni ileri-gecisten) ---
import torch
model.eval(); set_graft_mode(model,'teacher_forcing'); set_distill_backward(model, None)
energy={m.layer_idx: 0.0 for m in _grafted}
mse={m.layer_idx: 0.0 for m in _grafted}
nb_used=0
hooks=[]
def _mk(layer_idx):
    def hook(mod, inp, out):
        y = out[0] if isinstance(out, tuple) else out
        energy[layer_idx]  += y.detach().float().pow(2).mean().item()
        mse[layer_idx]     += float(mod.last_distill_loss)
    return hook
for m in _grafted: hooks.append(m.register_forward_hook(_mk(m.layer_idx)))
with torch.no_grad():
    for b in range(EVAL_BATCHES):
        s=b*SEQ
        if s+SEQ>len(val_ids): break
        model(val_ids[s:s+SEQ].unsqueeze(0).to(DEV))
        nb_used+=1
for h in hooks: h.remove()
NMSE={i: mse[i]/max(energy[i],1e-12) for i in mse}
RAWMSE={i: mse[i]/nb_used for i in mse}
order=sorted(NMSE, key=lambda i: NMSE[i])
print(f'NMSE olculdu ({nb_used} held-out chunk). Ucuzdan pahaliya siralama:')
for r,i in enumerate(order):
    flag=' <-REF' if i in REF_LAYERS else ('  (ilk/son)' if i in (0,N_LAYERS-1) else '')
    print(f'  #{r+1:>2} katman {i:>2}: NMSE {NMSE[i]:.4f}{flag}')
json.dump({'model':MODEL_ID,'steps1':STEPS1,'seq':SEQ,'eval_chunks':nb_used,
           'nmse':NMSE,'raw_mse':RAWMSE,'order':order,'ref_layers':REF_LAYERS,'traj':traj},
          open(os.path.join(OUT_DIR,'layer_linearization_map.json'),'w'), indent=2)
print('JSON:', os.path.join(OUT_DIR,'layer_linearization_map.json'))

In [ ]:
# --- 6. GRAFIK + ON-KAYITLI HIPOTEZ HUKMU ---
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
import numpy as np
xs=list(range(N_LAYERS)); ys=[NMSE[i] for i in xs]
colors=['#c0392b' if i in REF_LAYERS else ('#7f8c8d' if i in (0,N_LAYERS-1) else '#2471a3') for i in xs]
fig,ax=plt.subplots(figsize=(13,4.5))
ax.bar(xs, ys, color=colors)
ax.set_xlabel('katman indeksi'); ax.set_ylabel('NMSE (dusuk = ucuz linearize)')
ax.set_title('Katman-basi linearize maliyeti — Qwen2.5-1.5B (kirmizi=mevcut referans 6, gri=ilk/son)')
ax.set_xticks(xs); ax.grid(axis='y', alpha=.3)
png=os.path.join(OUT_DIR,'layer_linearization_map.png')
fig.tight_layout(); fig.savefig(png, dpi=130, bbox_inches='tight'); print('Grafik:', png)

def mean(ids): return float(np.mean([NMSE[i] for i in ids]))
cheap6  = order[:6]; cheap13 = order[:13]
ref6_m  = mean(REF_LAYERS); cheap6_m = mean(cheap6)
odd13   = [i for i in range(1,N_LAYERS-1) if i%2==1]   # §22a 13-katman ucurum seti
print('\n=== ON-KAYITLI HUKUM ===')
print(f'H1 (secim manevrasi): en-ucuz-6 {cheap6} ortalama NMSE {cheap6_m:.4f} vs '
      f'referans-6 {REF_LAYERS} {ref6_m:.4f}')
_gain=100*(ref6_m-cheap6_m)/ref6_m if ref6_m>0 else 0
if _gain>=20:
    print(f'  -> en-ucuz-6, referanstan %{_gain:.0f} daha ucuz (>=%20): SECIM MANEVRASI VAR. '
          f'Ilkeli secim daha yuksek yogunlugu destekleyebilir. Faz-B onerilir.')
elif _gain<=-20:
    print(f'  -> referans-6 aslinda daha ucuz (%{-_gain:.0f}): tek-indeks sansli iyi bir secimmis.')
else:
    print(f'  -> fark kucuk (%{_gain:.0f}): tek-indeks ~ ilkeli secim; kazanc yogunlukta, secimde degil.')
first_last=[0,N_LAYERS-1]; rank={i:r for r,i in enumerate(order)}
_fl_expensive=all(rank[i]>=N_LAYERS-8 for i in first_last)
print(f'H2 (ilk/son pahali mi): katman 0 sira #{rank[0]+1}, katman {N_LAYERS-1} sira #{rank[N_LAYERS-1]+1} '
      f'(28 icinde) -> {"DOGRULANDI (en pahali 8 icinde)" if _fl_expensive else "beklenenden ucuz -> metrigi sorgula"}')
print(f'\nYogunluk-itme adayi (en-ucuz-13): {sorted(cheap13)}  ort NMSE {mean(cheap13):.4f}')
print(f'  vs §22a ucurum seti (odd-13) {odd13} ort NMSE {mean(odd13):.4f}')
print('\nDURUST KISIT: NMSE bir PROXY (tek-katman, temiz girdi). Birikimli hata ve '
      'PPL duyarliligini yakalamaz. Bu harita adaylari SIRALAR; gercek kazanc Faz-B ile '
      '(en-ucuz-K graft -> tam egitim -> PPL vs §22a) dogrulanir.')